#  Gerador de Dados
Este notebook simula partidas válidas de Jogo da Velha e gera os conjuntos de Treino, Validação e Teste em arquivos CSV. Isso garante que todos os algoritmos do trabalho sejam testados exatamente sob as mesmas condições.

In [11]:
import pandas as pd
import numpy as np
import random
import os
from sklearn.model_selection import train_test_split

In [ ]:
# Lógica de Jogo Real
def verificar_estado(tab):
    vitorias = [[0,1,2], [3,4,5], [6,7,8], [0,3,6], [1,4,7], [2,5,8], [0,4,8], [2,4,6]]
    for v in vitorias:
        soma = tab[v[0]] + tab[v[1]] + tab[v[2]]
        if soma == 3: return 'X vence'
        if soma == -3: return 'O vence'
    for v in vitorias:
        soma = tab[v[0]] + tab[v[1]] + tab[v[2]]
        if abs(soma) == 2 and 0 in [tab[v[0]], tab[v[1]], tab[v[2]]]:
            return 'Possibilidade de Fim de Jogo'
    if 0 not in tab: return 'Empate'
    return 'Tem jogo'

def simular_partidas(n_partidas):
    estados = []
    for _ in range(n_partidas):
        tab = [0] * 9
        jog = 1
        pos = list(range(9))
        random.shuffle(pos)
        for p in pos:
            tab[p] = jog
            est = verificar_estado(tab)
            estados.append(tab.copy() + [est])
            if est in ['X vence', 'O vence', 'Empate']: break
            jog *= -1
    return estados

In [13]:
# Geração e Balanceamento
print("Simulando partidas... Aguarde.")
dados_brutos = simular_partidas(50000)
df = pd.DataFrame(dados_brutos, columns=[f'pos{i}' for i in range(9)] + ['target'])

df_bal = df.sample(frac=1, random_state=42).groupby('target').head(2500).reset_index(drop=True)

print("Distribuição das classes no dataset balanceado:")
print(df_bal['target'].value_counts())

Simulando partidas... Aguarde.
Distribuição das classes no dataset balanceado:
target
Tem jogo                        2500
Possibilidade de Fim de Jogo    2500
X vence                         2500
Empate                          2500
O vence                         2500
Name: count, dtype: int64


In [14]:
# Divisão Estratificada
X = df_bal.drop(columns=['target'])
y = df_bal['target']

x_train, x_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
x_test, x_val, y_test, y_val = train_test_split(x_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp)

train_df = pd.concat([x_train, y_train], axis=1)
val_df = pd.concat([x_val, y_val], axis=1)
test_df = pd.concat([x_test, y_test], axis=1)

print(f"Dataset de Treino: {train_df.shape}")
print(f"Dataset de Validação: {val_df.shape}")
print(f"Dataset de Teste: {test_df.shape}")

Dataset de Treino: (8750, 10)
Dataset de Validação: (1875, 10)
Dataset de Teste: (1875, 10)


In [15]:
# Salvamento dos Arquivos
if not os.path.exists('data'):
    os.makedirs('data')

train_df.to_csv('data/train.csv', index=False)
val_df.to_csv('data/val.csv', index=False)
test_df.to_csv('data/test.csv', index=False)

print("Sucesso! Arquivos salvos em 'data/train.csv', 'data/val.csv' e 'data/test.csv'.")
print("Agora todos os seus modelos (KNN, MLP, etc) usarão a mesma base de comparação.")

Sucesso! Arquivos salvos em 'data/train.csv', 'data/val.csv' e 'data/test.csv'.
Agora todos os seus modelos (KNN, MLP, etc) usarão a mesma base de comparação.
